# Frozen DistilBERT encodings with logistic regression

This notebook documents **binary classification** of campaign outcomes (successful vs failed). The **training** and **validation** tables under `data/features/` are used as given; no additional random split is applied.

A **frozen** `distilbert-base-uncased` encoder produces **768-dimensional** mean-pooled sentence vectors for each campaign text. A **logistic regression** classifier is fit on top of either (1) those vectors alone or (2) their **concatenation** with a fixed **tabular feature vector** built from the same rows.


In [17]:
import pandas as pd
import numpy as np
import torch

from tqdm import tqdm
from scipy.sparse import hstack, csr_matrix

from transformers import AutoTokenizer, AutoModel

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


## Data and labels

Predictor column names for the tabular block are read from `data/features/features_no_scale.txt` and `data/features/features_scale.txt`.

Observations are restricted to `successful` or `failed`. The text field concatenates `name` and `blurb`. The target is coded **1** for successful campaigns and **0** otherwise. Rows with missing text or label are dropped.


In [18]:
def load_feature_names(path):
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]

FEATURES_NO_SCALE = load_feature_names("data/features/features_no_scale.txt")
FEATURES_SCALE = load_feature_names("data/features/features_scale.txt")

def prepare_aligned(df):
    df = df[df["state"].isin(["successful", "failed"])].copy()
    df["text"] = df["name"].fillna("") + " " + df["blurb"].fillna("")
    df["label"] = (df["state"] == "successful").astype(int)
    df = df.dropna(subset=["text", "label"]).reset_index(drop=True)
    return df

train_aligned = prepare_aligned(pd.read_csv("data/features/train.csv"))
val_aligned = prepare_aligned(pd.read_csv("data/features/val.csv"))

train_df = train_aligned[["text", "label"]]
val_df = val_aligned[["text", "label"]]

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
train_df.head()


Train shape: (66096, 2)
Val shape: (22032, 2)


,text,label
0,Bring Art ala Carte back to the community! Art...,0
1,The PRESENT MIND Deck Mentalist Grant Price de...,1
2,Zeus-X Pro ⚡ - The World's Most Futuristic Uni...,1
3,Make 100 : The Spotting Chart Project - WWII A...,1
4,Art & Hue | Graphic Pop Art | Bespoke & Custom...,1


## Frozen encoder

The DistilBERT **encoder weights are not updated** in this notebook: `AutoModel` is placed in evaluation mode and embedding extraction runs under `torch.no_grad()`. The downstream logistic regression is the only component estimated from the training labels.


In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name).to(device)
bert_model.eval()


Using device: cpu


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

## Mean pooling

Token representations from the last hidden state are averaged with weights given by the attention mask (padding tokens excluded). The result is one vector per document.


In [20]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


## Text embedding extraction

Each split is encoded in mini-batches (`padding=True`, `truncation=True`, `max_length=128`). Vectors are stacked as dense `numpy` arrays of shape **(n, 768)**.


In [21]:
def get_frozen_embeddings(texts, batch_size=32, max_length=128):
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i : i + batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = bert_model(**inputs)
            pooled = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])

        all_embeddings.append(pooled.cpu().numpy())

    return np.vstack(all_embeddings)

X_train_text = get_frozen_embeddings(train_df["text"].tolist(), batch_size=32, max_length=128)
X_val_text = get_frozen_embeddings(val_df["text"].tolist(), batch_size=32, max_length=128)

print("Train text embeddings shape:", X_train_text.shape)
print("Val text embeddings shape:", X_val_text.shape)


100%|██████████| 689/689 [02:13<00:00,  5.15it/s]

Train text embeddings shape: (66096, 768)
Val text embeddings shape: (22032, 768)


## Tabular feature matrix

The no-scale and scale column groups are concatenated. `StandardScaler` is **fit on the training partition only** and applied to the validation partition. `nan_to_num` replaces non-finite values.


In [22]:
scaler = StandardScaler()

train_no_scale = train_aligned[FEATURES_NO_SCALE].astype(np.float32).to_numpy()
val_no_scale = val_aligned[FEATURES_NO_SCALE].astype(np.float32).to_numpy()

train_scaled = scaler.fit_transform(train_aligned[FEATURES_SCALE].astype(np.float32))
val_scaled = scaler.transform(val_aligned[FEATURES_SCALE].astype(np.float32))

X_train_struct = np.hstack([train_no_scale, train_scaled]).astype(np.float32)
X_val_struct = np.hstack([val_no_scale, val_scaled]).astype(np.float32)

X_train_struct = np.nan_to_num(X_train_struct, nan=0.0, posinf=0.0, neginf=0.0)
X_val_struct = np.nan_to_num(X_val_struct, nan=0.0, posinf=0.0, neginf=0.0)

print("Train structured shape:", X_train_struct.shape)
print("Val structured shape:", X_val_struct.shape)


Train structured shape: (66096, 29)
Val structured shape: (22032, 29)


## Design matrices for the two classifiers

**Model A:** sparse matrix of shape **(n, 768)** containing frozen text embeddings only.

**Model B:** horizontal concatenation of frozen text embeddings with the **tabular** matrix (**768 + 29 = 797** columns), stored as a sparse matrix for compatibility with `LogisticRegression` in `scikit-learn`.


In [23]:
X_train_frozen_only = csr_matrix(X_train_text)
X_val_frozen_only = csr_matrix(X_val_text)

X_train_frozen_plus = hstack([csr_matrix(X_train_text), csr_matrix(X_train_struct)])
X_val_frozen_plus = hstack([csr_matrix(X_val_text), csr_matrix(X_val_struct)])

y_train = train_df["label"].values
y_val = val_df["label"].values

print("Frozen text only — train:", X_train_frozen_only.shape, "val:", X_val_frozen_only.shape)
print("Frozen text + tabular — train:", X_train_frozen_plus.shape, "val:", X_val_frozen_plus.shape)


Frozen text only — train: (66096, 768) val: (22032, 768)
Frozen text + tabular — train: (66096, 797) val: (22032, 797)


## Logistic regression

The same routine fits `LogisticRegression` (`max_iter=2000`, fixed `random_state`) on the training design matrix and reports accuracy, F1, precision, recall, and ROC AUC on the **validation** partition, together with a classification report and confusion matrix.


In [24]:
def evaluate_logreg(X_train, X_val, y_train, y_val, model_name):
    clf = LogisticRegression(max_iter=2000, random_state=42)

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)
    y_score = clf.predict_proba(X_val)[:, 1]

    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred)
    rec = recall_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_score)

    print(f"\n{model_name} results")
    print(f"Accuracy : {acc:.4f}")
    print(f"F1       : {f1:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"ROC AUC  : {auc:.4f}")

    print("\nClassification report:")
    print(classification_report(y_val, y_pred))

    print("\nConfusion matrix:")
    print(confusion_matrix(y_val, y_pred))

    return {
        "Model": model_name,
        "accuracy": acc,
        "f1": f1,
        "precision": prec,
        "recall": rec,
        "roc_auc": auc,
    }


## Fitted models and validation scores

Two models are estimated: **frozen text embeddings only**, and **frozen text embeddings with tabular features**.


In [25]:
frozen_only_results = evaluate_logreg(
    X_train_frozen_only,
    X_val_frozen_only,
    y_train,
    y_val,
    "Frozen DistilBERT (text embeddings only)",
)

frozen_plus_results = evaluate_logreg(
    X_train_frozen_plus,
    X_val_frozen_plus,
    y_train,
    y_val,
    "Frozen DistilBERT + tabular features",
)



Frozen DistilBERT (text embeddings only) results
Accuracy : 0.7331
F1       : 0.7861
Precision: 0.7492
Recall   : 0.8268
ROC AUC  : 0.7975

Classification report:
              precision    recall  f1-score   support

           0       0.70      0.60      0.65      8964
           1       0.75      0.83      0.79     13068

    accuracy                           0.73     22032
   macro avg       0.73      0.71      0.72     22032
weighted avg       0.73      0.73      0.73     22032


Confusion matrix:
[[ 5347  3617]
 [ 2264 10804]]

Frozen DistilBERT + tabular features results
Accuracy : 0.7765
F1       : 0.8181
Precision: 0.7906
Recall   : 0.8477
ROC AUC  : 0.8474

Classification report:
              precision    recall  f1-score   support

           0       0.75      0.67      0.71      8964
           1       0.79      0.85      0.82     13068

    accuracy                           0.78     22032
   macro avg       0.77      0.76      0.76     22032
weighted avg       0.77    

## Comparison

The table below summarizes **validation** accuracy, F1, precision, recall, and ROC AUC. Rows are sorted by **F1** (descending).


In [26]:
comparison_df = pd.DataFrame([frozen_only_results, frozen_plus_results])
comparison_df = comparison_df[["Model", "accuracy", "f1", "precision", "recall", "roc_auc"]]
comparison_df = comparison_df.sort_values("f1", ascending=False).reset_index(drop=True)
comparison_df

,Model,accuracy,f1,precision,recall,roc_auc
0,Frozen DistilBERT + tabular features,0.776462,0.818138,0.790552,0.847720,0.847385
1,Frozen DistilBERT (text embeddings only),0.733070,0.786060,0.749185,0.826752,0.797462
